# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LeylaAghayeva1/ml-search-engineering/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [24]:
from dotenv import load_dotenv
from pathlib import Path
import os

# Find the project root (two levels above work/notebooks)
project_root = Path.cwd().parents[1]
env_path = project_root / ".env"


load_dotenv(env_path)

HF_TOKEN = os.getenv("HF_TOKEN")

print("Token found:", HF_TOKEN is not None)

    
from pathlib import Path

Token found: True


In [4]:
# List all available tables in the Hugging Face warehouse
import duckdb

con = duckdb.connect()

con.execute(
    f"CREATE SECRET (TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

rel = "hf://datasets/FlyRank/internship-warehouse"

# con.sql(f"""
# SELECT COUNT(*)
# FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
# """).df()
# con.sql(f"""
# SELECT *
# FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
# LIMIT 5
# """).df()
con.sql(f"""
DESCRIBE
SELECT *
FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
""").df()

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


## 1. Unit of analysis + time window

One row represents the daily performance snapshot of one content item for one client.

The grain of the dataset is:

(client_hash_id, content_hash_id, report_date)

Each row contains SEO, analytics, AI traffic, and engagement measurements collected for a specific content item on a specific date.

The dataset contains historical daily observations over the available reporting period.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date
FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,first_date,last_date
0,78835655,2025-01-27,2026-06-30


In [9]:
con.sql(f"""
SELECT
    COUNT(DISTINCT client_hash_id) AS unique_clients,
    COUNT(DISTINCT content_hash_id) AS unique_content_items
FROM (
    SELECT *
    FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
    LIMIT 100000
)
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,unique_clients,unique_content_items
0,4,7611


In [10]:
con.sql(f"""
SELECT
    COUNT(*) AS sampled_rows,
    COUNT(DISTINCT (
        client_hash_id,
        content_hash_id,
        report_date
    )) AS unique_daily_records
FROM (
    SELECT *
    FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
    LIMIT 100000
)
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,sampled_rows,unique_daily_records
0,100000,100000


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*
## Features

The following fields represent measurable performance signals that can be used as model inputs:

### Search performance
- gsc_impressions
- gsc_clicks
- gsc_sum_position
- gsc_avg_position

### Analytics performance
- ga4_pageviews
- ga4_sessions
- ga4_users
- ga4_engaged_sessions
- ga4_total_engagement_sec

### Traffic sources
- sessions_organic
- sessions_direct
- sessions_referral
- sessions_social
- sessions_paid
- sessions_ai

### AI traffic sources
- ai_chatgpt
- ai_perplexity
- ai_gemini
- ai_copilot
- ai_claude
- ai_meta
- ai_other

### Engagement
- scroll_events


## Label

No explicit prediction label exists in this table.

A label such as content performance direction (for example:
- up
- stable
- down

) would need to be derived from changes in performance metrics over time.


## Context

These fields describe the observation but are not direct predictive signals:

- report_date
- month
- client_has_gsc
- client_has_ga4
- gsc_data_available
- ga4_data_available


## Excluded

Identifier fields are excluded from modeling:

- client_hash_id
- content_hash_id

These fields identify entities but do not represent measurable content behavior.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
con.sql(f"""
DESCRIBE
SELECT *
FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
""").df()

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*
The assumptions above are verified through checks of:

- dataset size
- uniqueness of the analysis grain
- missing identifiers
- available date range

In [11]:
# This cell is for CODE (numbers, a query, a check).

con.sql(f"""
SELECT
    COUNT(*) AS sampled_rows,
    COUNT(DISTINCT client_hash_id) AS unique_clients,
    COUNT(DISTINCT content_hash_id) AS unique_content_items,
    COUNT(DISTINCT (
        client_hash_id,
        content_hash_id,
        report_date
    )) AS unique_daily_records
FROM (
    SELECT *
    FROM read_parquet(
        '{rel}/fact_content_daily_performance/**/*.parquet'
    )
    LIMIT 10000
)
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,sampled_rows,unique_clients,unique_content_items,unique_daily_records
0,10000,3,3521,10000


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*
This dataset describes observed content performance but cannot prove causal relationships.

Important limitations:

- The data shows correlations between performance signals and outcomes, not why performance changed.
- External factors such as algorithm updates, competition, seasonality, and market changes are not included.
- Daily observations from the same content item are related because they come from the same historical timeline.
- Some clients may have incomplete GSC or GA4 data availability.
- A future prediction target must be carefully defined because the current table does not contain an explicit label.
- Identifier columns cannot be used as meaningful predictive features because they only represent entity identity.
- During exploration, some remote parquet partitions produced decompression errors when performing full scans. Therefore, validation queries were limited to sampled records instead of complete scans.

In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
con.sql(f"""
SELECT
    COUNT(*) AS sampled_rows,

    COUNT(*) FILTER (
        WHERE client_hash_id IS NULL
    ) AS missing_clients,

    COUNT(*) FILTER (
        WHERE content_hash_id IS NULL
    ) AS missing_content,

    COUNT(*) FILTER (
        WHERE report_date IS NULL
    ) AS missing_dates

FROM (
    SELECT *
    FROM read_parquet(
        '{rel}/fact_content_daily_performance/**/*.parquet'
    )
    LIMIT 10000
)
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,sampled_rows,missing_clients,missing_content,missing_dates
0,10000,0,0,0


In [14]:
con.sql(f"""
SELECT
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date
FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,first_date,last_date
0,2025-01-27,2026-06-30


In [15]:
con.sql(f"""
SELECT
    COUNT(*) AS total_rows,

    COUNT(*) FILTER (
        WHERE client_has_gsc = FALSE
    ) AS rows_without_gsc,

    COUNT(*) FILTER (
        WHERE client_has_ga4 = FALSE
    ) AS rows_without_ga4

FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,rows_without_gsc,rows_without_ga4
0,78835655,98006,31965740


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.